# Population and data audit

This notebook freezes the population contract used by every subsequent analysis. It reproduces the manuscript's DINS–LARIAC linkage summary, validates the height-completed building scenes, distinguishes assessed and exposed structures from the larger geometric network, and writes machine-readable audit tables.

**Manuscript links:** Methods, *Data and Population Sample* and *Scene Reconstruction*; Extended Data population-flow and height-imputation tables. Fire-perimeter membership is descriptive and is not an eligibility criterion.

In [1]:
import sys
from pathlib import Path

cwd = Path.cwd().resolve()
candidates = [cwd, cwd.parent, cwd.parent.parent]
PACKAGE_ROOT = next(
    (path for path in candidates
     if (path / 'data').exists() and (path / 'src').exists()
     and (path / 'notebooks').exists()),
    None,
)
if PACKAGE_ROOT is None:
    raise RuntimeError('Run from the reproduction-package root or a notebook subdirectory')
PROJECT_ROOT = PACKAGE_ROOT
sys.path.insert(0, str(PACKAGE_ROOT))

import pandas as pd
from IPython.display import display
from src.analysis.population_audit import build_population_audit

DATA = PROJECT_ROOT / 'data'
RESULTS = PACKAGE_ROOT / 'results'
print('package:', PACKAGE_ROOT)
print('data:', DATA)

package: .
data: ./data


## Build and validate the frozen population

The audit fails if building identifiers are duplicated, final heights are miseng or non-positive, the fire population changes, or the `exposed` flag differs from the definition $F^*>0$.

In [2]:
audit = build_population_audit(DATA, RESULTS)
print('Wrote audit outputs to', RESULTS)

Wrote audit outputs to ./results


## Linkage and analytical population

In [3]:
display(audit['population_flow'])
display(audit['dins_linkage_distance'].round(3))
display(audit['analysis_outcomes'])
display(audit['analysis_exposure'])

,stage,buildings_or_records,definition
0,DINS records for Eaton and Palisades,30492,Source DINS records before spatial linkage
1,DINS records linked within 50 m,30401,Nearest LARIAC centroid within the prespecifie...
2,Unique closest building matches,28307,Closest DINS record retained when inspections ...
3,Eligible assessed structures,28208,Linked to the exposure table with an accessibl...
4,Primary exposed analysis population,25127,Eligible assessed structures with F* > 0


,unique_matches,median_m,p90_m,p95_m,p99_m,maximum_m
0,28307,2.387,5.344,6.751,11.035,49.328


fire,EATON,PALISADES,POOLED
outcome,,,
no_damage,7259,3921,11180
partial,1000,921,1921
destroyed,8592,6515,15107


fire,EATON,PALISADES,POOLED
exposure,,,
zero_F_star,2256,825,3081
positive_F_star,14595,10532,25127


## Scene completeness and perimeter context

The height table verifies the reconstruction rule. The mesh table shows why the assessed and exposed populations cannot be inferred from the mapped fire perimeters alone.

In [4]:
display(audit['height_imputation'])
display(audit['mesh_population'])

,fire,buildings,lariac_height,nearby_median,one_storey_fallback,miseng_final_height
0,EATON,36967,25115,11639,213,0
1,PALISADES,16200,15999,201,0,0
2,POOLED,53167,41114,11840,213,0


,fire,mesh_buildings,inside_perimeter,outside_perimeter,radex_records,assessed,assessed_inside_perimeter,assessed_outside_perimeter,positive_F_star_all_buildings,positive_F_star_assessed,positive_F_star_unassessed
0,EATON,36967,12997,23970,36715,16851,11931,4920,16560,14595,1965
1,PALISADES,16200,9986,6214,15916,11357,8999,2358,11651,10532,1119
2,POOLED,53167,22983,30184,52631,28208,20930,7278,28211,25127,3084


## Publication checksum

These assertions are the gate for all downstream notebooks. If any fails, manuscript estimates and figures must not be regenerated until the population change is reconciled.

In [5]:
population = audit['manifest']['analysis_population']
assert population['assessed'] == 28_208
assert population['positive_F_star'] == 25_127
assert population['by_fire']['EATON'] == {'assessed': 16_851, 'positive_F_star': 14_595}
assert population['by_fire']['PALISADES'] == {'assessed': 11_357, 'positive_F_star': 10_532}
print('PASS: canonical population = 28,208 assessed; 25,127 with F* > 0')
print('Manifest:', RESULTS / 'analysis_manifest.json')

PASS: canonical population = 28,208 assessed; 25,127 with F* > 0
Manifest: ./results/analysis_manifest.json


## Outputs

The `results/` directory now contains the population flow, linkage-distance summary, outcomes by fire, exposure eligibility by fire, height provenance, mesh/perimeter context and a checksummed JSON analysis manifest. Subsequent notebooks must read the same frozen inputs and reproduce these population checks.